<a href="https://colab.research.google.com/github/sjkim-audio/Bass-separator/blob/main/notebooks/evaluation/05_Tempo_Diagnostics_and_Edge_Case_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
from google.colab import drive

print("📂 구글 드라이브 마운트 중...")
drive.mount('/content/drive')

print("\n🐍 [파이썬] 필수 분석 라이브러리 설치 중...")
!pip install -q pretty_midi librosa tqdm pandas
print("✅ 환경 설정 완료.")

In [ ]:
import glob
import json

# 대소문자 변형 가능성을 모두 커버하기 위한 드라이브 경로 탐색
DRIVE_RECORD_DIR = "/content/drive/MyDrive/Bass_separator/evaluation_record"
if not os.path.exists(DRIVE_RECORD_DIR):
    DRIVE_RECORD_DIR = "/content/drive/MyDrive/Bass_Separator/evaluation_record"

print(f"🔍 탐색 경로: {DRIVE_RECORD_DIR}")

search_pattern = os.path.join(DRIVE_RECORD_DIR, "E2E_*_batch_results.json")
json_paths = glob.glob(search_pattern)

if not json_paths:
    raise FileNotFoundError(f"❌ {search_pattern} 경로에서 JSON 파일을 찾을 수 없습니다. 경로를 확인하십시오.")

# 가장 최근 날짜의 JSON 파일 확정
LATEST_JSON_PATH = sorted(json_paths)[-1]
print(f"✅ 최신 JSON 파일 로드 준비 완료: {LATEST_JSON_PATH}")

In [ ]:
# I/O 병목을 막기 위해 드라이브의 zip을 Colab 로컬 디스크로 압축 해제
DRIVE_ZIP_PATH = "/content/drive/MyDrive/Bass_separator/dataset/slakh_test.zip"
LOCAL_TEST_DIR = "/content/slakh_processed/test"

os.makedirs(LOCAL_TEST_DIR, exist_ok=True)

print("🗜️ 데이터셋 압축 해제 중 (수 분 소요될 수 있습니다)...")
!unzip -q -o {DRIVE_ZIP_PATH} -d {LOCAL_TEST_DIR}

print("\n✅ 추출된 유효 트랙 수:")
!ls -1 {LOCAL_TEST_DIR} | wc -l

In [ ]:
import pretty_midi
import librosa
import numpy as np
import warnings
from tqdm import tqdm

warnings.filterwarnings('ignore', module='librosa')

with open(LATEST_JSON_PATH, 'r', encoding='utf-8') as f:
    results_data = json.load(f)

track_scores = []
for track_id, metrics in results_data.items():
    try:
        quantized_f1 = metrics.get('quantized', {}).get('Onset_Pitch_F1', 1.0)
        sdr = metrics.get('separation', {}).get('SDR', float('nan'))
        track_scores.append({"track_id": track_id, "f1_score": quantized_f1, "sdr": sdr})
    except AttributeError:
        continue

# F1 스코어 최하위 5개 트랙 선별
worst_5 = sorted(track_scores, key=lambda x: x['f1_score'])[:5]
worst_5_ids = [row['track_id'] for row in worst_5]

bpm_errors = []
half_double_cases = []
failure_cases = []

print(f"🚀 총 {len(track_scores)}개 트랙 BPM 교차 검증 시작 (약 2~4분 소요)...\n")

for row in tqdm(track_scores, desc="BPM 분석 진행률"):
    track_id = row['track_id']
    track_folder = os.path.join(LOCAL_TEST_DIR, track_id)

    gt_midi_path = os.path.join(track_folder, "bass.mid")
    mix_audio_path = os.path.join(track_folder, "mix.flac")
    if not os.path.exists(mix_audio_path):
        mix_audio_path = os.path.join(track_folder, "mix.wav")

    gt_bpm, est_bpm = -1.0, -1.0

    # GT BPM 추출
    if os.path.exists(gt_midi_path):
        try:
            tempi = pretty_midi.PrettyMIDI(gt_midi_path).get_tempo_changes()[1]
            if len(tempi) > 0: gt_bpm = float(tempi[0])
        except Exception: pass

    # Est BPM 추출 (오디오 30초 로드)
    if os.path.exists(mix_audio_path):
        try:
            y, sr = librosa.load(mix_audio_path, sr=None, duration=30.0)
            tempo = librosa.beat.beat_track(y=y, sr=sr)[0]
            est_bpm = float(tempo[0] if isinstance(tempo, np.ndarray) else tempo)
        except Exception: pass

    # 최하위 5개 트랙인 경우 결과 상세 출력용 저장
    for w in worst_5:
        if w['track_id'] == track_id:
            w['gt_bpm'] = gt_bpm
            w['est_bpm'] = est_bpm

    # 전체 템포 오차 분류 트리
    if gt_bpm > 0 and est_bpm > 0:
        error = abs(gt_bpm - est_bpm)
        bpm_errors.append(error)

        if error > 3.0:
            ratio = est_bpm / gt_bpm
            if (0.45 <= ratio <= 0.55) or (1.90 <= ratio <= 2.10):
                half_double_cases.append({"track": track_id, "gt": gt_bpm, "est": est_bpm})
            elif error >= 10.0:
                failure_cases.append({"track": track_id, "gt": gt_bpm, "est": est_bpm})

# --- 결과 출력 ---
if bpm_errors:
    print("\n==================================================")
    print("🎯 전체 트랙 BPM 추적 성능 결과")
    print("==================================================")
    print(f"✅ 평균 오차 (MAE)    : {np.mean(bpm_errors):.2f} BPM")
    print(f"✅ 정확도 (±3 BPM내)  : {(len([e for e in bpm_errors if e <= 3.0]) / len(bpm_errors)) * 100:.2f} %")
    print(f"🚨 반/배수 에러       : {len(half_double_cases)} 건")
    print(f"🚨 엇박 추적 실패     : {len(failure_cases)} 건")

print("\n==================================================")
print("🚨 [F1 Score 최하위 5개 트랙 정밀 분석]")
print("==================================================")
for row in worst_5:
    f1 = row['f1_score'] * 100 if row['f1_score'] <= 1.0 else row['f1_score']
    bpm_err_str = f"{abs(row['gt_bpm'] - row['est_bpm']):.1f} BPM" if row['gt_bpm'] > 0 and row['est_bpm'] > 0 else "N/A"
    print(f"🎵 {row['track_id']} | Quantized F1: {f1:.2f}% | SDR: {row['sdr']:.2f}dB")
    print(f"   ┗ 🥁 GT BPM: {row['gt_bpm']:.1f} ➔ Est BPM: {row['est_bpm']:.1f} (오차: {bpm_err_str})")